In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, HBox, VBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ------------------------------------------------------------
# 1. DESCRIPTION
# ------------------------------------------------------------

description = HTML("""
<div style="
    border:1px solid #b9d7f5;
    border-radius:8px;
    padding:8px 10px;
    margin-bottom:10px;
    font-size:13px;
    line-height:1.35;
    background-color:#f7fbff;
">
<div><b>Purpose:</b> Demonstrate how cascading elementary low-pass sections produces a higher-order filter.</div>
<div><b>Construction:</b> N identical first-order low-pass sections are connected in cascade.</div>
<div><b>Normalization:</b> The pole frequency of each section is adjusted so that the composite filter always has its −3 dB point at ωc.</div>
<div><b>What we see:</b> Increasing the filter order increases the stopband attenuation rate and modifies the phase and group delay.</div>
</div>
""")

# ------------------------------------------------------------
# 2. CONTROLS
# ------------------------------------------------------------

order_slider = IntSlider(min=1, max=10, step=1, value=1, description='Order N:', continuous_update=True, style={'description_width':'70px'}, layout=Layout(width='250px'))

wc_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=1.0, description='ωc:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'70px'}, layout=Layout(width='250px'))

# ------------------------------------------------------------
# 3. LEGEND
# ------------------------------------------------------------

legend_html = HTML("""
<div style="
    border:1px solid #cccccc;
    border-radius:5px;
    padding:7px 9px;
    width:150px;
    font-size:13px;
    line-height:1.7;
    background:white;
">
<div><span style="display:inline-block; width:32px; border-top:3px solid red; vertical-align:middle; margin-right:7px;"></span>Composite filter</div>
<div><span style="display:inline-block; width:32px; border-top:2px dashed gray; vertical-align:middle; margin-right:7px;"></span>Single section</div>
<div><span style="display:inline-block; width:32px; border-top:2px dotted black; vertical-align:middle; margin-right:7px;"></span>ωc</div>
</div>
""")

parameter_label = HTML("<div style='font-size:14px; font-weight:bold; margin-top:12px; margin-bottom:4px;'>Filter Parameters:</div>")

# ------------------------------------------------------------
# 4. INFORMATION FRAME
# ------------------------------------------------------------

info_html = HTML()

# ------------------------------------------------------------
# 5. CREATE MAGNITUDE FIGURE ONCE
# ------------------------------------------------------------

fig_mag, ax_mag = plt.subplots(figsize=(6.6, 2.4))

ax_mag.set_xscale('log')

single_mag_line, = ax_mag.plot([], [], linestyle='--', linewidth=1.5, alpha=0.65)
composite_mag_line, = ax_mag.plot([], [], 'r-', linewidth=2.2)

wc_mag_line = ax_mag.axvline(1.0, color='black', linestyle=':', linewidth=1.4)
minus3_line = ax_mag.axhline(-3.0103, color='gray', linestyle='--', linewidth=1.0)
cutoff_point, = ax_mag.plot([1.0], [-3.0103], 'ko', markersize=4)

ax_mag.set_xlim(0.05, 100.0)
ax_mag.set_ylim(-100.0, 5.0)

ax_mag.set_xlabel('Angular Frequency ω (rad/s)', fontsize=11)
ax_mag.set_ylabel('Magnitude (dB)', fontsize=11)

ax_mag.set_title('Higher-Order Low-Pass Filter — Magnitude Response', fontsize=13, fontweight='bold', pad=7)

ax_mag.grid(True, which='both', linestyle=':', alpha=0.35)

fig_mag.subplots_adjust(left=0.13, right=0.98, bottom=0.20, top=0.88)

fig_mag.canvas.header_visible = False
fig_mag.canvas.toolbar_visible = False
fig_mag.canvas.resizable = False
fig_mag.canvas.layout.margin = '0px 0px -8px 0px'

# ------------------------------------------------------------
# 6. CREATE PHASE FIGURE ONCE
# ------------------------------------------------------------

fig_phase, ax_phase = plt.subplots(figsize=(6.6, 2.25))

ax_phase.set_xscale('log')

single_phase_line, = ax_phase.plot([], [], linestyle='--', linewidth=1.5, alpha=0.65)
composite_phase_line, = ax_phase.plot([], [], 'r-', linewidth=2.2)

wc_phase_line = ax_phase.axvline(1.0, color='black', linestyle=':', linewidth=1.4)
zero_phase_line = ax_phase.axhline(0.0, color='gray', linestyle='--', linewidth=1.0)

ax_phase.set_xlim(0.05, 100.0)
ax_phase.set_ylim(-100.0, 5.0)

ax_phase.set_xlabel('Angular Frequency ω (rad/s)', fontsize=11)
ax_phase.set_ylabel('Phase (degrees)', fontsize=11)

ax_phase.set_title('Higher-Order Low-Pass Filter — Phase Response', fontsize=13, fontweight='bold', pad=7)

ax_phase.grid(True, which='both', linestyle=':', alpha=0.35)

fig_phase.subplots_adjust(left=0.13, right=0.98, bottom=0.21, top=0.87)

fig_phase.canvas.header_visible = False
fig_phase.canvas.toolbar_visible = False
fig_phase.canvas.resizable = False
fig_phase.canvas.layout.margin = '0px 0px -8px 0px'

# ------------------------------------------------------------
# 7. CREATE GROUP-DELAY FIGURE ONCE
# ------------------------------------------------------------

fig_delay, ax_delay = plt.subplots(figsize=(6.6, 2.25))

ax_delay.set_xscale('log')

single_delay_line, = ax_delay.plot([], [], linestyle='--', linewidth=1.5, alpha=0.65)
composite_delay_line, = ax_delay.plot([], [], 'r-', linewidth=2.2)

wc_delay_line = ax_delay.axvline(1.0, color='black', linestyle=':', linewidth=1.4)

ax_delay.set_xlim(0.05, 100.0)
ax_delay.set_ylim(0.0, 2.0)

ax_delay.set_xlabel('Angular Frequency ω (rad/s)', fontsize=11)
ax_delay.set_ylabel('Group Delay (s)', fontsize=11)

ax_delay.set_title('Higher-Order Low-Pass Filter — Group Delay', fontsize=13, fontweight='bold', pad=7)

ax_delay.grid(True, which='both', linestyle=':', alpha=0.35)

fig_delay.subplots_adjust(left=0.13, right=0.98, bottom=0.21, top=0.87)

fig_delay.canvas.header_visible = False
fig_delay.canvas.toolbar_visible = False
fig_delay.canvas.resizable = False

# ------------------------------------------------------------
# 8. UPDATE FUNCTION
# ------------------------------------------------------------

def update_higher_order_filter(change=None):

    N = order_slider.value
    wc = wc_slider.value

    # --------------------------------------------------------
    # NORMALIZED SECTION POLE
    # --------------------------------------------------------

    wp = wc / np.sqrt(2.0**(1.0 / N) - 1.0)

    # --------------------------------------------------------
    # FREQUENCY AXIS
    # --------------------------------------------------------

    omega = np.logspace(np.log10(wc / 20.0), np.log10(wc * 100.0), 4001)

    # --------------------------------------------------------
    # SINGLE FIRST-ORDER SECTION
    # --------------------------------------------------------

    H1 = 1.0 / (1.0 + 1j * omega / wp)

    # --------------------------------------------------------
    # CASCADE OF N IDENTICAL SECTIONS
    # --------------------------------------------------------

    H = H1**N

    magnitude1_db = 20.0 * np.log10(np.maximum(np.abs(H1), 1e-12))
    magnitude_db = 20.0 * np.log10(np.maximum(np.abs(H), 1e-12))

    phase1 = np.unwrap(np.angle(H1))
    phase = np.unwrap(np.angle(H))

    phase1_deg = np.rad2deg(phase1)
    phase_deg = np.rad2deg(phase)

    group_delay1 = -np.gradient(phase1, omega)
    group_delay = -np.gradient(phase, omega)

    # --------------------------------------------------------
    # UPDATE MAGNITUDE RESPONSE
    # --------------------------------------------------------

    single_mag_line.set_data(omega, magnitude1_db)
    composite_mag_line.set_data(omega, magnitude_db)

    wc_mag_line.set_xdata([wc, wc])
    cutoff_point.set_data([wc], [-3.0103])

    ax_mag.set_xlim(omega[0], omega[-1])
    ax_mag.set_ylim(-100.0, 5.0)

    # --------------------------------------------------------
    # UPDATE PHASE RESPONSE
    # --------------------------------------------------------

    single_phase_line.set_data(omega, phase1_deg)
    composite_phase_line.set_data(omega, phase_deg)

    wc_phase_line.set_xdata([wc, wc])

    ax_phase.set_xlim(omega[0], omega[-1])

    phase_min = min(np.min(phase_deg), np.min(phase1_deg))

    ax_phase.set_ylim(1.08 * phase_min, 5.0)

    # --------------------------------------------------------
    # UPDATE GROUP DELAY
    # --------------------------------------------------------

    single_delay_line.set_data(omega, group_delay1)
    composite_delay_line.set_data(omega, group_delay)

    wc_delay_line.set_xdata([wc, wc])

    ax_delay.set_xlim(omega[0], omega[-1])

    delay_max = max(np.max(group_delay), np.max(group_delay1))

    if delay_max <= 0.0:
        delay_max = 1.0

    ax_delay.set_ylim(0.0, 1.10 * delay_max)

    # --------------------------------------------------------
    # INFORMATION FRAME
    # --------------------------------------------------------

    slope = -20 * N
    phase_inf = -90 * N

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:7px 9px;
        margin-top:8px;
        font-size:12px;
        line-height:1.8;
        background:white;
        width:255px;
        box-sizing:border-box;
        white-space:nowrap;
    ">

        <div>
            <b>Order N:</b>
            <span style="color:#0066cc;">{N}</span>
        </div>

        <div>
            <b>Composite cutoff ωc:</b>
            <span style="color:#0066cc;">{wc:.2f} rad/s</span>
        </div>

        <div>
            <b>Section pole ωp:</b>
            <span style="color:#0066cc;">{wp:.3f} rad/s</span>
        </div>

        <div>
            <b>|H(jωc)|:</b>
            <span style="color:#0066cc;">−3.01 dB</span>
        </div>

        <div>
            <b>Asymptotic slope:</b>
            <span style="color:#0066cc;">{slope} dB/decade</span>
        </div>

        <div>
            <b>Final phase:</b>
            <span style="color:#0066cc;">{phase_inf}°</span>
        </div>

    </div>
    """

    fig_mag.canvas.draw_idle()
    fig_phase.canvas.draw_idle()
    fig_delay.canvas.draw_idle()

# ------------------------------------------------------------
# 9. CONNECT CONTROLS
# ------------------------------------------------------------

order_slider.observe(update_higher_order_filter, names='value')
wc_slider.observe(update_higher_order_filter, names='value')

# ------------------------------------------------------------
# 10. LEFT COLUMN
# ------------------------------------------------------------

control_column = VBox([legend_html, parameter_label, order_slider, wc_slider, info_html], layout=Layout(width='285px', min_width='285px', flex='0 0 285px', align_items='flex-start', padding='2px 0px 0px 4px', overflow='hidden'))

# ------------------------------------------------------------
# 11. RIGHT COLUMN
# ------------------------------------------------------------

right_column = VBox([fig_mag.canvas, fig_phase.canvas, fig_delay.canvas], layout=Layout(width='auto', min_width='0px', flex='1 1 auto', align_items='flex-start', overflow='hidden'))

# ------------------------------------------------------------
# 12. MAIN AREA
# ------------------------------------------------------------

main_area = HBox([control_column, right_column], layout=Layout(width='100%', max_width='100%', align_items='flex-start', justify_content='flex-start', overflow='hidden'))

# ------------------------------------------------------------
# 13. INITIALIZE DATA
# ------------------------------------------------------------

update_higher_order_filter()

# ------------------------------------------------------------
# 14. FINAL DISPLAY
# ------------------------------------------------------------

display(description)
display(main_area)